# IMDB Sentiment Analysis: RNNs vs. Basic Neural Networks vs. Traditional ML

**Dataset:** [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) (Kaggle), with 50,000 reviews labelled `positive` / `negative` (25k each).

**Goal:** use a recurrent neural network (RNN) to classify review sentiment, then compare it with
1. **Traditional machine learning:** Logistic Regression, Linear SVM, and Multinomial Naive Bayes on TF-IDF features
2. **Basic (feed-forward) neural networks:** an MLP on TF-IDF features, and an embedding-bag network (averaged word embeddings + a dense layer)
3. **Recurrent neural networks:** a vanilla RNN, an LSTM, and a bidirectional GRU, all trained on word-index sequences

Every model uses the **same train/validation/test split**, and all results are reported on the held-out test set.

## Contents
1. Setup & data loading
2. Exploratory data analysis
3. Text cleaning & train/val/test split
4. Traditional ML (TF-IDF + LR / SVM / NB)
5. Basic neural networks (MLP, EmbeddingBag)
6. Recurrent neural networks (RNN, LSTM, BiGRU)
7. Comparison & discussion

## 1. Setup & data loading

In [ ]:
import os, re, time, random, json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
set_seed()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(os.cpu_count() or 1)
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
print("torch", torch.__version__, "| device:", DEVICE)

The notebook looks for the dataset in this order:
1. `data/IMDB Dataset.csv` (download the zip from the Kaggle page and extract it into `data/`)
2. the Kaggle notebook input path `/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv`
3. `kagglehub.dataset_download(...)`, which needs Kaggle API credentials

In [ ]:
def find_dataset():
    candidates = [Path("data/IMDB Dataset.csv"),
                  Path("/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")]
    for p in candidates:
        if p.exists():
            return p
    import kagglehub  # pip install kagglehub
    return Path(kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")) / "IMDB Dataset.csv"

df = pd.read_csv(find_dataset())
print(df.shape)
df.head()

## 2. Exploratory data analysis

In [ ]:
print(df["sentiment"].value_counts())
print("Missing values:", df.isna().sum().sum())
print("Duplicate reviews:", df["review"].duplicated().sum())

df["n_words"] = df["review"].str.split().str.len()
print(df["n_words"].describe().round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df["sentiment"].value_counts().plot.bar(ax=axes[0], color=["#4c72b0", "#dd8452"], rot=0)
axes[0].set_title("Class balance")
sns.histplot(data=df[df.n_words < 1200], x="n_words", hue="sentiment", bins=60, ax=axes[1])
axes[1].axvline(256, color="k", ls="--", lw=1, label="RNN max length (256)")
axes[1].set_title("Review length (words)")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "eda.png", dpi=120); plt.show()

The classes are perfectly balanced, so **accuracy** is a fair headline metric. Reviews are long (median ≈ 174 words, with a long tail),
which matters for the RNNs: they have to carry information across hundreds of time steps.
The dataset contains 418 duplicate reviews. We drop them **before** splitting so the same text can't appear in both the train and test sets.

## 3. Text cleaning & train/val/test split

In [ ]:
TAG_RE = re.compile(r"<[^>]+>")
TOKEN_RE = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[!?]")

def clean(text):
    text = TAG_RE.sub(" ", text.lower())          # strip "<br />" etc.
    return " ".join(TOKEN_RE.findall(text))       # keep words, contractions, ! and ?

df = df.drop_duplicates("review").reset_index(drop=True)
df["text"] = df["review"].map(clean)
df["label"] = (df["sentiment"] == "positive").astype(int)

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df.label, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df.label, random_state=SEED)
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")
print("\nExample cleaned text:\n", train_df.text.iloc[0][:300])

y_train, y_val, y_test = train_df.label.values, val_df.label.values, test_df.label.values

Stop-words are **kept** on purpose. Words such as *not*, *no*, and *but* flip sentiment, and the sequence models can use them.

We use an 80 / 10 / 10 split. The validation set is used for early stopping and model selection, and the test set is touched only for the final evaluation.

### Shared evaluation helper

In [ ]:
results = {}       # model name -> metrics dict
predictions = {}   # model name -> (y_pred, y_score)

def evaluate(name, y_true, y_pred, y_score, train_time, infer_time, n_params=None, family=""):
    m = dict(family=family,
             accuracy=accuracy_score(y_true, y_pred),
             precision=precision_score(y_true, y_pred),
             recall=recall_score(y_true, y_pred),
             f1=f1_score(y_true, y_pred),
             roc_auc=roc_auc_score(y_true, y_score),
             train_time_s=train_time,
             infer_time_s=infer_time,
             params=n_params)
    results[name] = m
    predictions[name] = (y_pred, y_score)
    print(f"{name:<22} acc={m['accuracy']:.4f}  f1={m['f1']:.4f}  auc={m['roc_auc']:.4f}  "
          f"train={train_time:.1f}s")
    return m

## 4. Traditional machine learning: TF-IDF + linear models

Each review becomes a sparse TF-IDF vector over unigrams and bigrams. Bigrams capture short phrases like *"not good"* and *"waste of"*,
which a pure bag-of-words loses. These models ignore word order beyond two-word windows.

In [ ]:
t0 = time.time()
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=200_000, sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(train_df.text)
X_val_tfidf = tfidf.transform(val_df.text)
X_test_tfidf = tfidf.transform(test_df.text)
tfidf_time = time.time() - t0
print(f"TF-IDF matrix: {X_train_tfidf.shape}, built in {tfidf_time:.1f}s")

A small grid over the regularisation strength `C` (or `alpha` for NB) is searched on the **validation** set.

In [ ]:
def tune_and_eval(name, make_model, grid, score_fn):
    best = None
    for p in grid:
        model = make_model(p).fit(X_train_tfidf, y_train)
        acc = accuracy_score(y_val, model.predict(X_val_tfidf))
        print(f"  {name} param={p:<6} val_acc={acc:.4f}")
        if best is None or acc > best[0]:
            best = (acc, p)
    # refit best on train and time it
    t0 = time.time(); model = make_model(best[1]).fit(X_train_tfidf, y_train); train_t = time.time() - t0
    t0 = time.time(); y_pred = model.predict(X_test_tfidf); score = score_fn(model, X_test_tfidf); inf_t = time.time() - t0
    n_params = int(np.prod(model.coef_.shape)) if hasattr(model, "coef_") else int(np.prod(model.feature_log_prob_.shape))
    evaluate(name, y_test, y_pred, score, train_t, inf_t, n_params, family="Traditional ML")
    return model

logreg = tune_and_eval("Logistic Regression",
                       lambda C: LogisticRegression(C=C, max_iter=2000, solver="liblinear"),
                       [1, 4, 10, 30], lambda m, X: m.predict_proba(X)[:, 1])
svm = tune_and_eval("Linear SVM",
                    lambda C: LinearSVC(C=C),
                    [0.1, 0.3, 1, 3], lambda m, X: m.decision_function(X))
nb = tune_and_eval("Naive Bayes",
                   lambda a: MultinomialNB(alpha=a),
                   [0.03, 0.1, 0.3, 1.0], lambda m, X: m.predict_proba(X)[:, 1])

Because the logistic-regression model is linear, we can read off which n-grams push a prediction positive or negative:

In [ ]:
feat = np.array(tfidf.get_feature_names_out())
coef = logreg.coef_[0]
top = pd.DataFrame({"most negative": feat[np.argsort(coef)[:15]],
                    "most positive": feat[np.argsort(coef)[::-1][:15]]})
top

## 5. Basic (feed-forward) neural networks

Two non-recurrent networks:

* **MLP on TF-IDF.** The same sparse features as the linear models (top 20k uni+bigrams) go through `Linear(20k→256) → ReLU → Dropout → Linear(256→1)`.
  This is a "basic neural network" in the classic sense: a fully connected network with one hidden layer.
* **EmbeddingBag (averaged word embeddings).** The model learns a 128-d embedding per word, **averages** the embeddings over the review, and feeds the result to a small dense head.
  It sees the same token sequences as the RNNs, but it ignores word order completely, which makes it the natural baseline for an RNN.

### Shared PyTorch training loop
Every network uses binary cross-entropy (on logits), the Adam optimizer, gradient clipping, and **early stopping on validation accuracy**: we keep the best epoch's weights.

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

@torch.no_grad()
def predict_scores(model, loader):
    model.eval()
    out = []
    for batch in loader:
        *inputs, _ = [b.to(DEVICE) for b in batch]
        out.append(torch.sigmoid(model(*inputs)).cpu())
    return torch.cat(out).numpy()

def train_model(name, model, train_loader, val_loader, test_loader, epochs=10, lr=1e-3,
                weight_decay=0.0, patience=2, clip=1.0, family=""):
    set_seed()
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.BCEWithLogitsLoss()
    history, best_acc, best_state, bad = [], -1, None, 0
    t_start = time.time()
    for epoch in range(1, epochs + 1):
        model.train(); t0 = time.time(); tot, n, correct = 0.0, 0, 0
        for batch in train_loader:
            *inputs, y = [b.to(DEVICE) for b in batch]
            opt.zero_grad()
            logits = model(*inputs)
            loss = loss_fn(logits, y.float())
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            tot += loss.item() * len(y); n += len(y); correct += ((logits > 0).long() == y).sum().item()
        val_scores = predict_scores(model, val_loader)
        val_loss = loss_fn(torch.logit(torch.tensor(val_scores).clamp(1e-7, 1 - 1e-7)), torch.tensor(y_val).float()).item()
        val_acc = accuracy_score(y_val, val_scores > 0.5)
        history.append(dict(epoch=epoch, train_loss=tot / n, train_acc=correct / n, val_loss=val_loss, val_acc=val_acc))
        print(f"[{name}] epoch {epoch:2d}  train_loss={tot/n:.4f} train_acc={correct/n:.4f}  "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}  ({time.time()-t0:.0f}s)")
        if val_acc > best_acc:
            best_acc, bad = val_acc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                print(f"[{name}] early stopping; best val_acc={best_acc:.4f}")
                break
    train_time = time.time() - t_start
    model.load_state_dict(best_state)
    t0 = time.time(); scores = predict_scores(model, test_loader); infer_time = time.time() - t0
    evaluate(name, y_test, (scores > 0.5).astype(int), scores, train_time, infer_time,
             count_params(model), family=family)
    histories[name] = pd.DataFrame(history)
    return model

histories = {}

### 5.1 MLP on TF-IDF features

In [ ]:
tfidf_nn = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9, max_features=20_000, sublinear_tf=True)
Xtr_nn = tfidf_nn.fit_transform(train_df.text).astype(np.float32)
Xva_nn = tfidf_nn.transform(val_df.text).astype(np.float32)
Xte_nn = tfidf_nn.transform(test_df.text).astype(np.float32)

class SparseDataset(Dataset):
    # keeps the matrix sparse; rows are densified per-batch in the collate fn
    def __init__(self, X, y): self.X, self.y = X, y
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, i): return i

def sparse_loader(X, y, shuffle, bs=256):
    def collate(idx):
        return torch.from_numpy(X[idx].toarray()), torch.from_numpy(y[idx])
    return DataLoader(SparseDataset(X, y), batch_size=bs, shuffle=shuffle, collate_fn=collate)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=256, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

mlp = train_model("MLP (TF-IDF)", MLP(Xtr_nn.shape[1]),
                  sparse_loader(Xtr_nn, y_train, True), sparse_loader(Xva_nn, y_val, False),
                  sparse_loader(Xte_nn, y_test, False), epochs=10, lr=1e-3, weight_decay=1e-5,
                  family="Basic NN")

### Tokenisation for the sequence models
The EmbeddingBag model and the RNNs share a vocabulary of the 30k most frequent training-set words (index 0 = padding, 1 = unknown).
Reviews are capped at **256 tokens**, which covers the full review for about 70% of the data. When a review is longer we keep its **last** 256 tokens,
because reviewers often state their verdict at the end. Sequences are padded per batch, and we pass the true lengths to the RNNs (`pack_padded_sequence`) so padding never reaches the recurrent state.

In [ ]:
MAX_LEN, VOCAB_SIZE = 256, 30_000
counter = Counter(tok for text in train_df.text for tok in text.split())
itos = ["<pad>", "<unk>"] + [w for w, _ in counter.most_common(VOCAB_SIZE - 2)]
stoi = {w: i for i, w in enumerate(itos)}

def encode(text):
    ids = [stoi.get(t, 1) for t in text.split()]
    return ids[-MAX_LEN:] if ids else [1]

seq_train = [encode(t) for t in train_df.text]
seq_val = [encode(t) for t in val_df.text]
seq_test = [encode(t) for t in test_df.text]
lens = np.array([len(s) for s in seq_train])
print(f"vocab={len(itos)}  coverage of train tokens={sum(counter[w] for w in itos[2:]) / sum(counter.values()):.3f}  "
      f"reviews not truncated={np.mean(lens < MAX_LEN):.2%}")

class SeqDataset(Dataset):
    def __init__(self, seqs, y): self.seqs, self.y = seqs, y
    def __len__(self): return len(self.seqs)
    def __getitem__(self, i): return self.seqs[i], self.y[i]

def pad_collate(batch):
    seqs, ys = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = torch.zeros(len(seqs), lengths.max(), dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :len(s)] = torch.tensor(s)
    return padded, lengths, torch.tensor(ys)

def seq_loader(seqs, y, shuffle, bs=64):
    return DataLoader(SeqDataset(seqs, y), batch_size=bs, shuffle=shuffle, collate_fn=pad_collate)

train_loader = seq_loader(seq_train, y_train, True)
val_loader, test_loader = seq_loader(seq_val, y_val, False), seq_loader(seq_test, y_test, False)

### 5.2 EmbeddingBag: averaged word embeddings + dense layer (order-agnostic)

In [ ]:
class EmbeddingBagNet(nn.Module):
    def __init__(self, vocab, emb=128, hidden=64, dropout=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(emb, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x, lengths):
        mean = self.emb(x).sum(1) / lengths.unsqueeze(1).to(x.device)   # padding embeds are zero
        return self.head(mean).squeeze(-1)

embbag = train_model("EmbeddingBag NN", EmbeddingBagNet(len(itos)), train_loader, val_loader, test_loader,
                     epochs=15, lr=1e-3, patience=3, family="Basic NN")

## 6. Recurrent neural networks

All three RNNs have the same skeleton:

`Embedding(30k, 128) → recurrent layer, 64 hidden units (packed sequence) → final hidden state → Dropout → Linear → logit`

(64 hidden units keep CPU training to a few minutes per epoch; on a GPU you can raise it to 128–256.)

| Model | Recurrent layer | Purpose |
|---|---|---|
| **Vanilla RNN** | `nn.RNN(128, 64)`, tanh | the textbook RNN; it shows the vanishing-gradient problem on long reviews |
| **LSTM** | `nn.LSTM(128, 64)` | gates and a cell state let information survive across many time steps |
| **BiGRU** | `nn.GRU(128, 64, bidirectional=True)`, 1 layer | reads the review in both directions; the two final states are concatenated |

Because we pass true lengths through `pack_padded_sequence`, the "final hidden state" is the state at the **last real token** of each review, not the state after the padding.

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab, cell="lstm", emb=128, hidden=64, bidirectional=False, dropout=0.4):
        super().__init__()
        self.emb = nn.Embedding(vocab, emb, padding_idx=0)
        rnn_cls = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}[cell]
        self.rnn = rnn_cls(emb, hidden, batch_first=True, bidirectional=bidirectional)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * (2 if bidirectional else 1), 1)

    def forward(self, x, lengths):
        e = self.drop(self.emb(x))
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, h = self.rnn(packed)
        if isinstance(h, tuple):          # LSTM returns (h_n, c_n)
            h = h[0]
        h = torch.cat([h[-2], h[-1]], dim=1) if self.rnn.bidirectional else h[-1]
        return self.fc(self.drop(h)).squeeze(-1)

### 6.1 Vanilla RNN

In [ ]:
vanilla_rnn = train_model("Vanilla RNN", RNNClassifier(len(itos), "rnn"), train_loader, val_loader, test_loader,
                          epochs=10, lr=1e-3, patience=3, family="RNN")

### 6.2 LSTM

In [ ]:
lstm = train_model("LSTM", RNNClassifier(len(itos), "lstm"), train_loader, val_loader, test_loader,
                   epochs=8, lr=1e-3, patience=2, family="RNN")

### 6.3 Bidirectional GRU

In [ ]:
bigru = train_model("BiGRU", RNNClassifier(len(itos), "gru", bidirectional=True), train_loader, val_loader, test_loader,
                    epochs=8, lr=1e-3, patience=2, family="RNN")

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name, h in histories.items():
    line, = axes[0].plot(h.epoch, h.train_loss, "--", alpha=0.6)
    axes[0].plot(h.epoch, h.val_loss, "-o", color=line.get_color(), ms=3, label=name)
    axes[1].plot(h.epoch, h.val_acc, "-o", ms=3, label=name)
axes[0].set(title="Loss (dashed = train, solid = validation)", xlabel="epoch", ylabel="BCE loss")
axes[1].set(title="Validation accuracy", xlabel="epoch", ylabel="accuracy")
axes[1].legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR / "training_curves.png", dpi=120); plt.show()

## 7. Comparison

In [ ]:
res = pd.DataFrame(results).T
res.index.name = "model"
res = res.sort_values("accuracy", ascending=False)
res.to_csv(RESULTS_DIR / "metrics.csv")
fmt = {c: "{:.4f}" for c in ["accuracy", "precision", "recall", "f1", "roc_auc"]}
fmt.update(train_time_s="{:.1f}", infer_time_s="{:.2f}", params="{:,.0f}")
res.style.format(fmt).background_gradient(subset=["accuracy", "f1", "roc_auc"], cmap="Greens")

In [ ]:
palette = {"Traditional ML": "#4c72b0", "Basic NN": "#dd8452", "RNN": "#55a868"}
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
order = res.sort_values("accuracy")
axes[0].barh(order.index, order.accuracy, color=order.family.map(palette))
for i, v in enumerate(order.accuracy):
    axes[0].text(v + 0.001, i, f"{v:.4f}", va="center", fontsize=9)
axes[0].set_xlim(order.accuracy.min() - 0.03, order.accuracy.max() + 0.015)
axes[0].set_title("Test accuracy")
axes[1].scatter(res.train_time_s, res.accuracy, c=res.family.map(palette), s=80)
for name, r in res.iterrows():
    axes[1].annotate(name, (r.train_time_s, r.accuracy), textcoords="offset points", xytext=(6, -3), fontsize=9)
axes[1].set_xscale("log"); axes[1].set(title="Accuracy vs. training time (CPU)", xlabel="training time [s, log]", ylabel="test accuracy")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in palette.values()]
axes[0].legend(handles, palette.keys(), loc="lower right")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "comparison.png", dpi=120); plt.show()

In [ ]:
names = list(res.index)
fig, axes = plt.subplots(2, (len(names) + 1) // 2, figsize=(4 * ((len(names) + 1) // 2), 7.5))
for ax, name in zip(axes.flat, names):
    cm = confusion_matrix(y_test, predictions[name][0])
    sns.heatmap(cm, annot=True, fmt="d", cbar=False, cmap="Blues", ax=ax,
                xticklabels=["neg", "pos"], yticklabels=["neg", "pos"])
    ax.set(title=f"{name}\nacc={res.loc[name, 'accuracy']:.4f}", xlabel="predicted", ylabel="true")
for ax in axes.flat[len(names):]:
    ax.axis("off")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "confusion_matrices.png", dpi=120); plt.show()

### Does review length matter?
We split the test set into length buckets and compare the best model of each family. Long reviews are where order-aware RNNs should help,
and also where truncation to 256 tokens and vanishing gradients hurt.

In [ ]:
test_len = test_df.text.str.split().str.len().values
buckets = pd.cut(test_len, [0, 100, 200, 400, 10_000], labels=["≤100", "101-200", "201-400", ">400"])
best_per_family = res.groupby("family")["accuracy"].idxmax().tolist() + ["Vanilla RNN"]
by_len = pd.DataFrame({name: pd.Series(predictions[name][0] == y_test).groupby(buckets, observed=True).mean()
                       for name in dict.fromkeys(best_per_family)})
by_len["n_reviews"] = pd.Series(buckets).value_counts().reindex(by_len.index).values
by_len.round(4)

### Where do the models disagree?

In [ ]:
best_rnn = res[res.family == "RNN"].index[0]
best_trad = res[res.family == "Traditional ML"].index[0]
p_rnn, p_trad = predictions[best_rnn][0], predictions[best_trad][0]
print(f"{best_rnn} right & {best_trad} wrong: {np.sum((p_rnn == y_test) & (p_trad != y_test))}")
print(f"{best_trad} right & {best_rnn} wrong: {np.sum((p_trad == y_test) & (p_rnn != y_test))}")
print(f"both wrong: {np.sum((p_rnn != y_test) & (p_trad != y_test))} / {len(y_test)}")

# simple ensemble: average of the two models' probabilities
p_lr = predictions["Logistic Regression"][1]
p_ens = (p_lr + predictions[best_rnn][1]) / 2
print(f"\nEnsemble (Logistic Regression + {best_rnn}) accuracy: {accuracy_score(y_test, p_ens > 0.5):.4f}")

In [ ]:
def predict_all(texts):
    cleaned = [clean(t) for t in texts]
    rows = {}
    Xt = tfidf.transform(cleaned)
    rows["Logistic Regression"] = logreg.predict_proba(Xt)[:, 1]
    rows["Linear SVM"] = 1 / (1 + np.exp(-svm.decision_function(Xt)))     # squashed margin, for display
    rows["MLP (TF-IDF)"] = torch.sigmoid(mlp(torch.from_numpy(tfidf_nn.transform(cleaned).astype(np.float32).toarray()).to(DEVICE))).detach().cpu().numpy()
    seqs = [encode(t) for t in cleaned]
    x, lengths, _ = pad_collate(list(zip(seqs, [0] * len(seqs))))
    for name, m in [("EmbeddingBag NN", embbag), ("Vanilla RNN", vanilla_rnn), ("LSTM", lstm), ("BiGRU", bigru)]:
        m.eval()
        with torch.no_grad():
            rows[name] = torch.sigmoid(m(x.to(DEVICE), lengths)).cpu().numpy()
    return pd.DataFrame(rows, index=texts).round(3)

examples = ["This movie was absolutely wonderful, the acting was superb and I loved every minute.",
            "A complete waste of time. The plot made no sense and the acting was terrible.",
            "I expected this to be bad, but it was actually not bad at all.",
            "The first half was great, but the ending ruined the whole thing.",
            "Not good."]
predict_all(examples)

### Discussion

*(The numbers below refer to the run saved in this notebook. Neural-network results move by a few tenths of a percent between runs and hardware.)*

**Overall ranking.** On this dataset all families land in a narrow band of roughly 87–91% accuracy. The **TF-IDF + linear models (Logistic Regression, Linear SVM) are among the strongest**, and the
best gated RNNs (LSTM / BiGRU) are competitive with them. They do **not** clearly beat the linear baselines, and they cost one to two orders of magnitude more training time.

**Why don't RNNs win outright?**
* IMDB reviews are long, and sentiment is mostly carried by **strong lexical cues** (*"waste"*, *"worst"*, *"7/10"*, *"excellent"*, *"must see"*). A bag of uni+bigrams with good regularisation captures most of that signal.
* The RNNs learn their embeddings **from scratch** on only 40k reviews. With pre-trained embeddings (GloVe, word2vec) or a pre-trained transformer (BERT gets ~94–95%), sequence models pull clearly ahead.
* RNNs read at most 256 tokens. About 30% of reviews are truncated, while TF-IDF sees the whole review.

**Vanilla RNN vs. gated RNNs.** The plain `tanh` RNN is the weakest model. It has to squeeze a 256-step review into one hidden state through repeated matrix multiplications, so gradients from the
loss vanish (or explode) before they reach early tokens. Its accuracy also drops on longer reviews (see the length table). The LSTM's and GRU's gates give an additive path through time, which fixes this
and gives a large jump in accuracy for almost the same number of parameters.

**Basic NN vs. RNN.** The **EmbeddingBag** network uses the same inputs and vocabulary as the RNNs but ignores word order. The gap between it and the LSTM/BiGRU measures what modelling *order* adds
(negation such as *"not bad"*, contrast such as *"…but the ending ruined it"*). The **MLP on TF-IDF** is close to logistic regression. A hidden layer on top of already very informative sparse features adds
little, and it overfits fast (validation loss rises after 1–2 epochs).

**Practical takeaway.**
| If you need… | Use |
|---|---|
| a strong, fast, interpretable baseline | TF-IDF + Logistic Regression / Linear SVM (trains in seconds on a CPU) |
| a model that understands word order / negation | LSTM or BiGRU, ideally with pre-trained embeddings |
| the best possible accuracy | a pre-trained transformer, fine-tuned |

The errors of the linear model and the RNN are only partly correlated, so simply averaging their probabilities gives a further small gain (see the ensemble cell above).